# Multiclass Classification Inference

This notebook shows how to use an already-trained multiclass sequence classifier with LociSimiles. The package does not train classifiers here; it runs inference, keeps the usual thresholdable `judgment_score`, and adds class metadata for labels such as `cit` and `cf`.

In [ ]:
# Reinstall from local source if you are editing the package in this repository.
# %pip install -e ..

In [ ]:
import os
from pathlib import Path

import pandas as pd

from locisimiles.document import Document
from locisimiles.evaluator import IntertextEvaluator
from locisimiles.pipeline import (
    CandidateJudge,
    ClassificationPipelineWithCandidateGeneration,
    pretty_print,
)

## Load the Example Corpus

In [ ]:
base_dir = Path.cwd()
if base_dir.name != "examples" and (base_dir / "examples").exists():
    base_dir = base_dir / "examples"

query_doc = Document(base_dir / "hieronymus_samples.csv", author="Hieronymus")
source_doc = Document(base_dir / "vergil_samples.csv", author="Vergil")

print(query_doc)
print(source_doc)

## Configure a Trained Multiclass Model

In [ ]:
label_names = ["no_match", "cit", "cf"]
positive_labels = ["cit", "cf"]

multiclass_model = os.environ.get("LOCISIMILES_MULTICLASS_MODEL", "").strip()
embedding_model = "julian-schelb/multilingual-e5-large-emb-lat-intertext-v1"
device = os.environ.get("LOCISIMILES_DEVICE", "cpu")

if multiclass_model:
    print(f"Using multiclass model: {multiclass_model}")
else:
    print(
        "Set LOCISIMILES_MULTICLASS_MODEL to a local path or HuggingFace model id ",
        "to run real multiclass classifier inference. The cells below still ",
        "demonstrate the output and evaluation format."
    )

In [ ]:
model_results = None
model_pipeline = None

if multiclass_model:
    model_pipeline = ClassificationPipelineWithCandidateGeneration(
        classification_name=multiclass_model,
        embedding_model_name=embedding_model,
        label_names=label_names,
        positive_labels=positive_labels,
        device=device,
    )
    model_results = model_pipeline.run(query=query_doc, source=source_doc, top_k=10)
    pretty_print(model_results)
else:
    print("Skipping model download/loading because no multiclass model was configured.")

## Inspect Class Metadata

In [ ]:
def rows_from_results(results):
    rows = []
    for query_id, judgments in results.items():
        for judgment in judgments:
            rows.append(
                {
                    "query_id": query_id,
                    "source_id": judgment.segment.id,
                    "judgment_score": judgment.judgment_score,
                    "predicted_label": judgment.predicted_label,
                    "class_probabilities": judgment.class_probabilities,
                }
            )
    return pd.DataFrame(rows)

if model_results is not None:
    rows_from_results(model_results).head(10)
else:
    print("No model results yet; the next section creates deterministic demo results.")

## Demo Results Without Downloading a Model

The small pipeline below returns handcrafted `CandidateJudge` objects with the same metadata a trained multiclass classifier would produce. This keeps the notebook runnable even when no local classifier is available.

In [ ]:
class DemoMulticlassPipeline:
    def run(self, query, source, top_k=10):
        source_by_id = {str(segment.id): segment for segment in source}
        return {
            "hier. adv. iovin. 1.41": [
                CandidateJudge(
                    segment=source_by_id["verg. aen. 11.508"],
                    candidate_score=0.94,
                    judgment_score=0.93,
                    predicted_class_id=1,
                    predicted_label="cit",
                    class_probabilities={"no_match": 0.07, "cit": 0.88, "cf": 0.05},
                ),
                CandidateJudge(
                    segment=source_by_id["verg. aen. 4.172"],
                    candidate_score=0.55,
                    judgment_score=0.18,
                    predicted_class_id=0,
                    predicted_label="no_match",
                    class_probabilities={"no_match": 0.82, "cit": 0.10, "cf": 0.08},
                ),
            ],
            "hier. adv. pelag. 1.23": [
                CandidateJudge(
                    segment=source_by_id["verg. ecl. 8.62"],
                    candidate_score=0.89,
                    judgment_score=0.84,
                    predicted_class_id=2,
                    predicted_label="cf",
                    class_probabilities={"no_match": 0.16, "cit": 0.12, "cf": 0.72},
                )
            ],
        }

demo_pipeline = DemoMulticlassPipeline()
demo_results = demo_pipeline.run(query_doc, source_doc, top_k=2)
pretty_print(demo_results)
rows_from_results(demo_results)

## Evaluate `cit` and `cf` Separately

In [ ]:
multiclass_ground_truth = pd.DataFrame(
    [
        {
            "query_id": "hier. adv. iovin. 1.41",
            "source_id": "verg. aen. 11.508",
            "label": "cit.",
        },
        {
            "query_id": "hier. adv. pelag. 1.23",
            "source_id": "verg. ecl. 8.62",
            "label": "cf.",
        },
    ]
)

demo_evaluator = IntertextEvaluator(
    query_doc=query_doc,
    source_doc=source_doc,
    ground_truth_csv=multiclass_ground_truth,
    pipeline=demo_pipeline,
    top_k=2,
    threshold=0.5,
)

demo_evaluator.evaluate_multiclass(labels=["cit", "cf"], strategy="argmax")

In [ ]:
demo_evaluator.evaluate_multiclass(labels=["cit", "cf"], strategy="thresholded")